# AI Career Advisor & Learning Path Generator  
## Colab-Ready Multi-Agent Career Advisory System using LangGraph + Groq

## Installations

In [1]:
!pip -q install -U groq langgraph ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 88.7 MB/s eta 0:00:00


## API Key Setup

In [2]:
import os

# Load from Colab Secrets ONLY
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError("❌ GROQ_API_KEY not found in Colab Secrets")

os.environ["GROQ_API_KEY"] = api_key

MODEL_NAME = "llama-3.3-70b-versatile"

print("✅ Groq API key loaded from secrets")
print("Model:", MODEL_NAME)

✅ Groq API key loaded from secrets
Model: llama-3.3-70b-versatile


## Imports

In [3]:
import json
import re
from typing import Any, Dict, List, TypedDict

from groq import Groq
from langgraph.graph import StateGraph, START, END
from IPython.display import Markdown, display

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

## Sample Profiles

In [4]:
student_profiles = [
    {"student_id": "S1", "profile_text": "I know Python basics and statistics, and I want to become a Data Scientist"},
    {"student_id": "S2", "profile_text": "I know HTML, CSS, JavaScript, and React basics, and I want to become a Full Stack Developer"}
]
student_profiles

[{'student_id': 'S1',
  'profile_text': 'I know Python basics and statistics, and I want to become a Data Scientist'},
 {'student_id': 'S2',
  'profile_text': 'I know HTML, CSS, JavaScript, and React basics, and I want to become a Full Stack Developer'}]

## Knowledge Base

In [5]:
ROLE_KNOWLEDGE_BASE = {
    "Data Scientist": {
        "required_skills": ["Python", "Statistics", "SQL", "Machine Learning", "Data Visualization", "Pandas", "NumPy", "Model Evaluation", "Feature Engineering", "Deployment"],
        "tools": ["Jupyter", "scikit-learn", "Pandas", "Matplotlib", "Git"],
        "projects": ["Exploratory data analysis project", "Predictive modeling project", "Mini deployment project with API or dashboard"]
    },
    "Data Analyst": {
        "required_skills": ["Excel", "SQL", "Python", "Statistics", "Data Cleaning", "Power BI", "Tableau", "Data Visualization"],
        "tools": ["Excel", "SQL", "Power BI", "Tableau", "Python"],
        "projects": ["Business dashboard", "Sales trend analysis", "EDA case study"]
    },
    "ML Engineer": {
        "required_skills": ["Python", "Machine Learning", "Deep Learning", "SQL", "APIs", "Docker", "Deployment", "Model Serving", "MLOps", "Git"],
        "tools": ["TensorFlow", "PyTorch", "FastAPI", "Docker", "GitHub"],
        "projects": ["Model serving API", "Image classifier", "End-to-end ML pipeline"]
    },
    "Full Stack Developer": {
        "required_skills": ["HTML", "CSS", "JavaScript", "React", "Node.js", "Express.js", "MongoDB", "REST APIs", "Authentication", "Git", "Deployment"],
        "tools": ["React", "Node.js", "Express.js", "MongoDB Atlas", "Postman", "Git", "Vercel"],
        "projects": ["MERN CRUD application", "Authentication-based web app", "Deployed full-stack project"]
    },
    "Frontend Developer": {
        "required_skills": ["HTML", "CSS", "JavaScript", "React", "Responsive Design", "State Management", "API Integration", "Git"],
        "tools": ["React", "Tailwind CSS", "Git", "Vercel", "Figma"],
        "projects": ["Responsive landing page", "API-driven dashboard", "UI clone project"]
    },
    "Backend Developer": {
        "required_skills": ["JavaScript", "Node.js", "Express.js", "REST APIs", "Databases", "Authentication", "Git", "Deployment"],
        "tools": ["Node.js", "Express.js", "Postman", "MongoDB", "Docker"],
        "projects": ["REST API project", "Authentication backend", "Scalable service mini project"]
    }
}

## State Object

In [6]:
class CareerAdvisorState(TypedDict, total=False):
    student_id: str
    profile_text: str
    extracted_skills: List[str]
    stated_goal: str
    level: str
    interests: List[str]
    profile_summary: str
    assessment: str
    recommended_roles: List[Dict[str, Any]]
    target_role: str
    skill_gaps: List[str]
    skill_gap_summary: str
    roadmap: Dict[str, Any]
    reasoning_summary: str
    final_advice: str
    route_label: str

## Helper Functions

In [7]:
def extract_json_block(text: str) -> Dict[str, Any]:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"```json\s*(\{.*?\})\s*```|```\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        return json.loads(match.group(1) or match.group(2))

    brace_match = re.search(r"\{.*\}", text, re.DOTALL)
    if brace_match:
        return json.loads(brace_match.group(0))

    raise ValueError(f"Could not parse JSON from model output:\n{text}")

def ask_llm_for_json(system_prompt: str, user_prompt: str) -> Dict[str, Any]:
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.2,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    raw_text = completion.choices[0].message.content.strip()
    return extract_json_block(raw_text)

## Agent Definitions

In [8]:
def profile_analyzer_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    system_prompt = '''
You are the Profile Analyzer Agent in a multi-agent career advisory system.
Return ONLY valid JSON:
{
  "extracted_skills": ["..."],
  "stated_goal": "...",
  "level": "beginner or intermediate or advanced",
  "interests": ["..."],
  "profile_summary": "...",
  "assessment": "..."
}
Rules:
- If the student says basics, treat them as beginner unless evidence suggests otherwise.
'''
    result = ask_llm_for_json(system_prompt, f"Student Profile: {state['profile_text']}")
    level = result.get("level", "beginner").strip().lower()
    if level not in {"beginner", "intermediate", "advanced"}:
        level = "beginner"
    return {
        **state,
        "extracted_skills": result.get("extracted_skills", []),
        "stated_goal": result.get("stated_goal", ""),
        "level": level,
        "interests": result.get("interests", []),
        "profile_summary": result.get("profile_summary", ""),
        "assessment": result.get("assessment", ""),
        "route_label": "beginner_path" if level == "beginner" else "intermediate_path"
    }

def career_recommendation_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    role_catalog = list(ROLE_KNOWLEDGE_BASE.keys())
    system_prompt = f'''
You are the Career Recommendation Agent.
Available role catalog:
{json.dumps(role_catalog, indent=2)}
Return ONLY valid JSON:
{{
  "recommended_roles": [
    {{"role": "...", "match_score": 0, "why_fit": "..."}}
  ],
  "target_role": "..."
}}
Rules:
- Use only roles from the catalog
- match_score must be 1 to 100
'''
    user_prompt = f'''
Profile summary: {state.get("profile_summary", "")}
Current skills: {state.get("extracted_skills", [])}
Goal: {state.get("stated_goal", "")}
Level: {state.get("level", "")}
Interests: {state.get("interests", [])}
'''
    result = ask_llm_for_json(system_prompt, user_prompt)
    return {**state, "recommended_roles": result.get("recommended_roles", []), "target_role": result.get("target_role", "")}

def skill_gap_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    target_role = state.get("target_role", "")
    current = {s.strip().lower() for s in state.get("extracted_skills", [])}
    required = ROLE_KNOWLEDGE_BASE.get(target_role, {}).get("required_skills", [])
    gaps = [skill for skill in required if skill.strip().lower() not in current]

    system_prompt = '''
You are the Skill Gap Agent.
Return ONLY valid JSON:
{"skill_gap_summary": "..."}
'''
    result = ask_llm_for_json(system_prompt, f"Current skills: {state.get('extracted_skills', [])}\nTarget role: {target_role}\nRequired skills: {required}\nSkill gaps: {gaps}")
    return {**state, "skill_gaps": gaps, "skill_gap_summary": result.get("skill_gap_summary", "")}

def beginner_learning_path_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    target_role = state.get("target_role", "")
    kb = ROLE_KNOWLEDGE_BASE.get(target_role, {})
    system_prompt = '''
You are the Learning Path Agent for a BEGINNER student.
Return ONLY valid JSON:
{
  "roadmap": {
    "beginner_stage": ["..."],
    "intermediate_stage": ["..."],
    "advanced_stage": ["..."],
    "suggested_projects": ["..."],
    "timeline_note": "...",
    "learning_strategy": "..."
  }
}
'''
    user_prompt = f"Level: {state.get('level')}\nTarget role: {target_role}\nCurrent skills: {state.get('extracted_skills', [])}\nSkill gaps: {state.get('skill_gaps', [])}\nTools: {kb.get('tools', [])}\nProjects: {kb.get('projects', [])}"
    result = ask_llm_for_json(system_prompt, user_prompt)
    return {**state, "roadmap": result.get("roadmap", {})}

def intermediate_learning_path_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    target_role = state.get("target_role", "")
    kb = ROLE_KNOWLEDGE_BASE.get(target_role, {})
    system_prompt = '''
You are the Learning Path Agent for an INTERMEDIATE or ADVANCED student.
Return ONLY valid JSON:
{
  "roadmap": {
    "beginner_stage": ["..."],
    "intermediate_stage": ["..."],
    "advanced_stage": ["..."],
    "suggested_projects": ["..."],
    "timeline_note": "...",
    "learning_strategy": "..."
  }
}
'''
    user_prompt = f"Level: {state.get('level')}\nTarget role: {target_role}\nCurrent skills: {state.get('extracted_skills', [])}\nSkill gaps: {state.get('skill_gaps', [])}\nTools: {kb.get('tools', [])}\nProjects: {kb.get('projects', [])}"
    result = ask_llm_for_json(system_prompt, user_prompt)
    return {**state, "roadmap": result.get("roadmap", {})}

def final_advisor_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    system_prompt = '''
You are the Final Advisor Agent.
Return ONLY valid JSON:
{
  "reasoning_summary": "...",
  "final_advice": "..."
}
'''
    user_prompt = f'''
Profile summary: {state.get("profile_summary", "")}
Assessment: {state.get("assessment", "")}
Recommended roles: {state.get("recommended_roles", [])}
Target role: {state.get("target_role", "")}
Skill gaps: {state.get("skill_gaps", [])}
Skill gap summary: {state.get("skill_gap_summary", "")}
Roadmap: {json.dumps(state.get("roadmap", {}), indent=2)}
'''
    result = ask_llm_for_json(system_prompt, user_prompt)
    return {**state, "reasoning_summary": result.get("reasoning_summary", ""), "final_advice": result.get("final_advice", "")}

## LangGraph Workflow

In [9]:
def route_after_gap_analysis(state: CareerAdvisorState) -> str:
    return state.get("route_label", "beginner_path")

graph_builder = StateGraph(CareerAdvisorState)
graph_builder.add_node("profile_analyzer", profile_analyzer_agent)
graph_builder.add_node("career_recommender", career_recommendation_agent)
graph_builder.add_node("skill_gap_agent", skill_gap_agent)
graph_builder.add_node("beginner_roadmap_agent", beginner_learning_path_agent)
graph_builder.add_node("intermediate_roadmap_agent", intermediate_learning_path_agent)
graph_builder.add_node("final_advisor", final_advisor_agent)

graph_builder.add_edge(START, "profile_analyzer")
graph_builder.add_edge("profile_analyzer", "career_recommender")
graph_builder.add_edge("career_recommender", "skill_gap_agent")
graph_builder.add_conditional_edges(
    "skill_gap_agent",
    route_after_gap_analysis,
    {
        "beginner_path": "beginner_roadmap_agent",
        "intermediate_path": "intermediate_roadmap_agent"
    }
)
graph_builder.add_edge("beginner_roadmap_agent", "final_advisor")
graph_builder.add_edge("intermediate_roadmap_agent", "final_advisor")
graph_builder.add_edge("final_advisor", END)

career_advisor_graph = graph_builder.compile()
print("LangGraph workflow compiled successfully.")

LangGraph workflow compiled successfully.


## Agent Flow Diagram

In [10]:
mermaid_diagram = '''
flowchart TD
    A[Student Input] --> B[Profile Analyzer Agent]
    B --> C[Career Recommendation Agent]
    C --> D[Skill Gap Agent]
    D --> E{Student Level?}
    E -->|Beginner| F[Beginner Learning Path Agent]
    E -->|Intermediate / Advanced| G[Accelerated Learning Path Agent]
    F --> H[Final Advisor Agent]
    G --> H[Final Advisor Agent]
    H --> I[Final Output]
'''
print(mermaid_diagram)


flowchart TD
    A[Student Input] --> B[Profile Analyzer Agent]
    B --> C[Career Recommendation Agent]
    C --> D[Skill Gap Agent]
    D --> E{Student Level?}
    E -->|Beginner| F[Beginner Learning Path Agent]
    E -->|Intermediate / Advanced| G[Accelerated Learning Path Agent]
    F --> H[Final Advisor Agent]
    G --> H[Final Advisor Agent]
    H --> I[Final Output]



## Output Rendering

In [11]:
def bullet_list(items):
    return "\n".join([f"- {item}" for item in items]) if items else "- Not available"

def render_roles(roles):
    if not roles:
        return "- No recommendations generated"
    lines = []
    for i, role in enumerate(roles, start=1):
        lines.append(
            f"{i}. **{role.get('role', 'Unknown')}** (Match Score: {role.get('match_score', 'N/A')})  \n"
            f"   - Why fit: {role.get('why_fit', '')}"
        )
    return "\n".join(lines)

def render_roadmap(roadmap):
    if not roadmap:
        return "Roadmap not generated."
    return f'''
### Beginner Stage
{bullet_list(roadmap.get("beginner_stage", []))}

### Intermediate Stage
{bullet_list(roadmap.get("intermediate_stage", []))}

### Advanced Stage
{bullet_list(roadmap.get("advanced_stage", []))}

### Suggested Projects
{bullet_list(roadmap.get("suggested_projects", []))}

### Timeline Note
{roadmap.get("timeline_note", "Not provided")}

### Learning Strategy
{roadmap.get("learning_strategy", "Not provided")}
'''

def display_final_result(result):
    report = f'''
# Final Advisory Result — {result.get("student_id", "Unknown")}

## 1. Student Profile Summary
**Original Input:** {result.get("profile_text", "")}

**Profile Summary:** {result.get("profile_summary", "")}

## 2. Current Skill Assessment
- **Extracted Skills:** {", ".join(result.get("extracted_skills", [])) or "Not available"}
- **Target Goal:** {result.get("stated_goal", "")}
- **Estimated Level:** {result.get("level", "").title()}
- **Assessment:** {result.get("assessment", "")}

## 3. Recommended Career Roles
{render_roles(result.get("recommended_roles", []))}

## 4. Skill Gaps
{bullet_list(result.get("skill_gaps", []))}

**Skill Gap Summary:** {result.get("skill_gap_summary", "")}

## 5. Personalized Learning Roadmap
{render_roadmap(result.get("roadmap", {}))}

## 6. Final Career Advice
{result.get("final_advice", "")}

## Reasoning Summary
{result.get("reasoning_summary", "")}
'''
    display(Markdown(report))

## Execute on 2 Profiles

In [12]:
all_results = []
for profile in student_profiles:
    result = career_advisor_graph.invoke({
        "student_id": profile["student_id"],
        "profile_text": profile["profile_text"]
    })
    all_results.append(result)
print("Execution complete.")

Execution complete.


In [13]:
for result in all_results:
    display_final_result(result)
    display(Markdown("---"))


# Final Advisory Result — S1

## 1. Student Profile Summary
**Original Input:** I know Python basics and statistics, and I want to become a Data Scientist

**Profile Summary:** The student has a foundation in Python programming and statistics, and is interested in pursuing a career in Data Science.

## 2. Current Skill Assessment
- **Extracted Skills:** Python, statistics
- **Target Goal:** Data Scientist
- **Estimated Level:** Beginner
- **Assessment:** The student is on the right track by having a basic understanding of Python and statistics. However, to become a Data Scientist, they will need to acquire more advanced skills in machine learning, data visualization, and data modeling.

## 3. Recommended Career Roles
1. **Data Analyst** (Match Score: 80)  
   - Why fit: The student has a foundation in statistics and is interested in data analysis, which are key skills for a Data Analyst role.
2. **Data Scientist** (Match Score: 90)  
   - Why fit: The student is interested in pursuing a career in Data Science and has a foundation in Python programming and statistics, which are essential skills for a Data Scientist role.
3. **ML Engineer** (Match Score: 70)  
   - Why fit: The student is interested in machine learning and has a foundation in Python programming, which are key skills for an ML Engineer role.

## 4. Skill Gaps
- SQL
- Machine Learning
- Data Visualization
- Pandas
- NumPy
- Model Evaluation
- Feature Engineering
- Deployment

**Skill Gap Summary:** To become a Data Scientist, you need to acquire skills in SQL, Machine Learning, Data Visualization, Pandas, NumPy, Model Evaluation, Feature Engineering, and Deployment, as you already possess Python and statistics skills.

## 5. Personalized Learning Roadmap

### Beginner Stage
- Learn SQL basics
- Introduction to Pandas and NumPy
- Fundamentals of Machine Learning
- Data Visualization with Matplotlib
- Exploratory data analysis project

### Intermediate Stage
- Model Evaluation techniques
- Feature Engineering methods
- Advanced Machine Learning concepts
- Predictive modeling project
- Data Visualization best practices

### Advanced Stage
- Hyperparameter tuning
- Model deployment strategies
- Advanced Data Visualization tools
- Mini deployment project with API or dashboard
- Collaboration and communication in Data Science

### Suggested Projects
- Exploratory data analysis project
- Predictive modeling project
- Mini deployment project with API or dashboard

### Timeline Note
Focus on building a strong foundation in the beginner stage (3-6 months), then progress to intermediate (6-9 months) and advanced stages (9-12 months)

### Learning Strategy
Practice with real-world projects, participate in Kaggle competitions, and engage with the Data Science community to stay updated with industry trends


## 6. Final Career Advice
Follow the provided roadmap, focusing on building a strong foundation in the beginner stage, then progressing to intermediate and advanced stages. Practice with real-world projects, participate in Kaggle competitions, and engage with the Data Science community to stay updated with industry trends and become a proficient Data Scientist.

## Reasoning Summary
The student has a solid foundation in Python programming and statistics, but needs to acquire advanced skills in machine learning, data visualization, and data modeling to become a Data Scientist. The recommended roadmap outlines a beginner, intermediate, and advanced stage to fill the skill gaps in SQL, Machine Learning, Data Visualization, Pandas, NumPy, Model Evaluation, Feature Engineering, and Deployment.


---


# Final Advisory Result — S2

## 1. Student Profile Summary
**Original Input:** I know HTML, CSS, JavaScript, and React basics, and I want to become a Full Stack Developer

**Profile Summary:** The student has a basic understanding of front-end development technologies and aims to become a full stack developer.

## 2. Current Skill Assessment
- **Extracted Skills:** HTML, CSS, JavaScript, React
- **Target Goal:** Full Stack Developer
- **Estimated Level:** Beginner
- **Assessment:** The student is on the right track by knowing the basics of HTML, CSS, JavaScript, and React. To become a full stack developer, they should focus on learning back-end development technologies such as Node.js, Express, and databases.

## 3. Recommended Career Roles
1. **Full Stack Developer** (Match Score: 90)  
   - Why fit: The student has a basic understanding of front-end development technologies and aims to become a full stack developer, which aligns with this role.
2. **Frontend Developer** (Match Score: 80)  
   - Why fit: The student has skills in HTML, CSS, JavaScript, and React, which are all relevant to front-end development.
3. **Backend Developer** (Match Score: 60)  
   - Why fit: Although the student's current skills are focused on front-end development, their interest in back-end development and goal of becoming a full stack developer suggest potential for growth in this area.

## 4. Skill Gaps
- Node.js
- Express.js
- MongoDB
- REST APIs
- Authentication
- Git
- Deployment

**Skill Gap Summary:** To become a Full Stack Developer, you need to acquire skills in Node.js, Express.js, MongoDB, REST APIs, Authentication, Git, and Deployment, as these are the areas where you have a skill gap.

## 5. Personalized Learning Roadmap

### Beginner Stage
- Learn Node.js basics
- Understand Express.js framework
- Get familiar with MongoDB and NoSQL databases
- Learn about REST APIs and API design
- Understand Authentication and Authorization concepts

### Intermediate Stage
- Build small-scale full-stack applications using MERN stack
- Implement Authentication and Authorization in applications
- Learn about Git version control and collaboration
- Deploy applications to cloud platforms like Vercel

### Advanced Stage
- Optimize and scale full-stack applications
- Learn about advanced MongoDB concepts and data modeling
- Implement caching, logging, and monitoring in applications
- Explore other full-stack frameworks and technologies

### Suggested Projects
- MERN CRUD application
- Authentication-based web app
- Deployed full-stack project
- Real-time chat application
- E-commerce website with payment gateway integration

### Timeline Note
Focus on building a strong foundation in the beginner stage (3-6 months), then move to intermediate stage (6-12 months), and finally advanced stage (1-2 years)

### Learning Strategy
Practice building small projects, participate in coding challenges, and learn from online resources like tutorials, blogs, and videos


## 6. Final Career Advice
Focus on building a strong foundation in the beginner stage, then move to intermediate and advanced stages, practicing with small projects, participating in coding challenges, and learning from online resources to become a proficient Full Stack Developer.

## Reasoning Summary
The student has a solid foundation in front-end development and aims to become a full stack developer. To achieve this goal, they need to acquire skills in back-end development technologies such as Node.js, Express, and databases. A tailored roadmap is provided to guide the student through beginner, intermediate, and advanced stages, with suggested projects and a recommended learning strategy.


---

## State Passing Demonstration

In [14]:
for result in all_results:
    compact_state = {
        "student_id": result.get("student_id"),
        "extracted_skills": result.get("extracted_skills"),
        "stated_goal": result.get("stated_goal"),
        "level": result.get("level"),
        "target_role": result.get("target_role"),
        "skill_gaps": result.get("skill_gaps"),
        "route_label": result.get("route_label"),
        "roadmap_keys": list(result.get("roadmap", {}).keys()) if result.get("roadmap") else []
    }
    print(json.dumps(compact_state, indent=2))
    print("=" * 80)

{
  "student_id": "S1",
  "extracted_skills": [
    "Python",
    "statistics"
  ],
  "stated_goal": "Data Scientist",
  "level": "beginner",
  "target_role": "Data Scientist",
  "skill_gaps": [
    "SQL",
    "Machine Learning",
    "Data Visualization",
    "Pandas",
    "NumPy",
    "Model Evaluation",
    "Feature Engineering",
    "Deployment"
  ],
  "route_label": "beginner_path",
  "roadmap_keys": [
    "beginner_stage",
    "intermediate_stage",
    "advanced_stage",
    "suggested_projects",
    "timeline_note",
    "learning_strategy"
  ]
}
{
  "student_id": "S2",
  "extracted_skills": [
    "HTML",
    "CSS",
    "JavaScript",
    "React"
  ],
  "stated_goal": "Full Stack Developer",
  "level": "beginner",
  "target_role": "Full Stack Developer",
  "skill_gaps": [
    "Node.js",
    "Express.js",
    "MongoDB",
    "REST APIs",
    "Authentication",
    "Git",
    "Deployment"
  ],
  "route_label": "beginner_path",
  "roadmap_keys": [
    "beginner_stage",
    "intermediate_

## Conclusion

This is a complete **multi-agent academic mini project** that uses:
- Groq LLM API
- LangGraph state workflow
- modular agent functions
- conditional routing for beginner vs intermediate learners
- structured output rendering

It is also extensible for future agents like:
- Resume Agent
- Interview Prep Agent
- Project Recommendation Agent
- Certification Advisor